# Day 4 v2 — Model 09: AITeamVN Improved (top-8, R-Drop, EMA 0.9999)

**Architecture:** `AITeamVN/Vietnamese_Embedding` (568M, 24L, 1024-dim, BGE-M3 base)
— partial unfreeze top **8**/24 layers → mean_pooling → price head + aux category head.

**Improvements vs NB06 (session 23 — research-backed):**
| Technique | NB06 (old) | NB09 (improved) |
|---|---|---|
| keep_top_layers | 4/24 (17%) | **8/24 (33%)** — optimal for 269K samples |
| EMA decay | 0.999 (window ~1K steps) | **0.9999** (window ~10K steps, 84K total) |
| warmup_ratio | 0.1 (full epoch ramp-up) | **0.05** (half epoch) |
| weight_decay | 0.02 | **0.01** (BERT standard) |
| llrd_decay | 0.9 | **0.85** (wider LR range for 8 layers) |
| batch_size | 32 | **24** (reduced for R-Drop 2x forward) |
| R-Drop | — | **alpha=0.3** (MSE consistency loss) |

## vast.ai Setup (chỉ chạy lần đầu)

```bash
pip install uv
uv sync
```

Restart kernel sau khi sync xong.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import json
import torch

from pricer_vi_2.items import Item
from pricer_vi_2.evaluator import evaluate, plot_training_history
from pricer_vi_2.bert_finetune_model import BERTFinetuneRunner

MODEL_NAME = "AITeamVN/Vietnamese_Embedding"
WEIGHT_DIR = Path("weights")
VAL_PRED_DIR = Path("val_predictions")

# Improved hyperparams (session 23)
KEEP_TOP      = 8       # top-8/24 layers = 33% encoder
BATCH         = 24      # reduced from 32 for R-Drop 2x forward
BASE_LR       = 2e-5
WEIGHT_DECAY  = 0.01    # was 0.02 — BERT standard
LLRD_DECAY    = 0.85    # was 0.9 — wider range for 8 layers
EPOCHS        = 10
PATIENCE      = 3
EMA_DECAY     = 0.9999  # was 0.999 — window ~10K steps for 84K total steps
WARMUP_RATIO  = 0.05    # was 0.1 — half epoch warmup
R_DROP_ALPHA  = 0.3     # R-Drop MSE consistency loss weight

print(f"torch: {torch.__version__} | cuda: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Load Data

In [ ]:
train, val, test = Item.from_hub("SeanSunny/items_tv_v9")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

## 2. Setup Runner

- Tokenize 269K train + 3926 val (stored as tensors, ~550MB for max_length=256)
- Freeze bottom 16/24 transformer layers, **unfreeze top 8**
- LLRD: head lr=2e-5, each lower trainable layer × decay=0.85
- Approx trainable params: ~30M encoder + ~1M heads = ~31M total

In [ ]:
runner = BERTFinetuneRunner(train, val)

runner.setup(
    model_name=MODEL_NAME,
    keep_top_layers=KEEP_TOP,
    batch_size=BATCH,
    max_length=256,
    base_lr=BASE_LR,
    weight_decay=WEIGHT_DECAY,
    llrd_decay=LLRD_DECAY,
    dropout=0.2,
)

## 3. Train

10 epochs, early stopping patience=3.
Val MAE evaluated on full 3926 samples per epoch using EMA model (ema_decay=0.9999).
R-Drop: 2x forward per step → effective batch=24 × 2 forward passes.

Expected: ~50-60 min/epoch on RTX 3090 Ti (269K samples, batch=24, top-8 layers).

In [ ]:
history = runner.train(
    epochs=EPOCHS,
    patience=PATIENCE,
    huber_delta=1.0,
    aux_alpha=0.1,
    ema_decay=EMA_DECAY,
    warmup_ratio=WARMUP_RATIO,
    max_grad_norm=1.0,
    r_drop_alpha=R_DROP_ALPHA,
)

## 4. Training History

In [ ]:
plot_training_history(history, title="AITeamVN Improved (top-8, R-Drop, EMA 0.9999)")

## 5. Save Weights + Val Predictions + Test Predictions

In [ ]:
WEIGHT_DIR.mkdir(exist_ok=True)
runner.save(str(WEIGHT_DIR / "aitvn_improved.pth"))
print("Saved weights/aitvn_improved.pth")

VAL_PRED_DIR.mkdir(exist_ok=True)

print("Running val predictions (3926 samples)...")
val_preds = runner.val_predictions()
with open(VAL_PRED_DIR / "aitvn_improved_val.json", "w") as f:
    json.dump(val_preds, f)
print(f"Saved val_predictions/aitvn_improved_val.json ({len(val_preds)} samples)")

print("Running test predictions (3872 samples)...")
test_preds = runner.test_predictions(test)
with open(VAL_PRED_DIR / "aitvn_improved_test.json", "w") as f:
    json.dump(test_preds, f)
print(f"Saved val_predictions/aitvn_improved_test.json ({len(test_preds)} samples)")

## 6. Evaluate on 200 Test Samples

In [ ]:
def aitvn_improved_pricer(item):
    return runner.inference(item)

results = evaluate(aitvn_improved_pricer, test)
print(f"MAE: {results['mae']:.1f}k VND | MSE: {results['mse']:,.0f} | R2: {results['r2']:.1f}%")

## 7. Sanity Check

In [ ]:
sample = test[0]
pred = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  {sample.price:.1f}k VND")
print(f"Predict: {pred:.1f}k VND")
print(f"Error:   {abs(pred - sample.price):.1f}k VND")

ckpt = torch.load(str(WEIGHT_DIR / "aitvn_improved.pth"), map_location="cpu", weights_only=False)
print(f"Checkpoint keys: {list(ckpt.keys())}")
print(f"keep_top_layers={ckpt['keep_top_layers']} | model_name={ckpt['model_name']}")